# Pattern 5: Hierarchical Task Decomposition

> **Google Cloud Reference**: [Hierarchical Task Decomposition](https://docs.cloud.google.com/architecture/choose-design-pattern-agentic-ai-system#hierarchical-task-decomposition-pattern)

A **root agent** decomposes a complex task into sub-tasks, delegates to **manager agents**, who further decompose to **worker agents**. This creates a multi-level tree of specialization.

```
                    [Root Agent]
                   /     |      \
          [Manager 1] [Manager 2] [Manager 3]
           /     \       |        /     \
       [W1]     [W2]   [W3]    [W4]   [W5]
```

**When to use:**
- Ambiguous, open-ended problems requiring multi-step reasoning
- Complex research, planning, and synthesis
- Tasks where decomposing ambiguity is the primary challenge

**Trade-offs:**
- ✅ Best for highly complex, ambiguous problems
- ✅ Comprehensive, high-quality results
- ⚠️ High latency (nested model calls)
- ⚠️ High cost (multiple levels of model orchestration)
- ⚠️ Difficult to debug

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

### Example: Research Project Orchestrator

```
Root: Research Director
 ├── Manager: Data Gathering Team
 │    ├── Worker: Web Researcher
 │    └── Worker: Statistics Collector
 ├── Manager: Analysis Team
 │    ├── Worker: Trend Analyst
 │    └── Worker: Competitive Analyst
 └── Manager: Synthesis Team
      └── Worker: Report Writer
```

In [2]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools

model = OpenAIChat(id="google/gemini-2.5-flash")

# ── Level 3: Worker Agents ─────────────────────────────────

web_researcher = Agent(
    name="Web Researcher",
    role="Search the web for current information and recent developments",
    model=model,
    tools=[DuckDuckGoTools()],
    instructions=[
        "Search for 3-5 credible sources.",
        "Focus on information from the last 12 months.",
        "Extract key facts, statistics, and quotes.",
    ],
    markdown=True,
)

statistics_collector = Agent(
    name="Statistics Collector",
    role="Find and validate key market statistics and data points",
    model=model,
    tools=[DuckDuckGoTools()],
    instructions=[
        "Focus exclusively on quantitative data: market size, growth rates, percentages.",
        "Always note the source and year of each statistic.",
        "Identify if statistics are projections vs. actual figures.",
    ],
    markdown=True,
)

trend_analyst = Agent(
    name="Trend Analyst",
    role="Identify and analyze emerging trends from gathered data",
    model=model,
    instructions=[
        "Identify 3-5 macro trends shaping the industry.",
        "Rate each trend: Early / Growing / Mature / Declining.",
        "Explain the business impact of each trend.",
    ],
    markdown=True,
)

competitive_analyst = Agent(
    name="Competitive Analyst",
    role="Analyze the competitive landscape and key players",
    model=model,
    instructions=[
        "Identify top 5 players in the space.",
        "For each: market position, key differentiator, and recent moves.",
        "Identify whitespace opportunities.",
    ],
    markdown=True,
)

report_writer = Agent(
    name="Report Writer",
    role="Synthesize all research into a polished executive report",
    model=model,
    instructions=[
        "Write an executive research report in clean Markdown.",
        "Structure: Executive Summary, Market Overview, Key Statistics, Trends, Competitive Landscape, Conclusions & Recommendations.",
        "Keep it concise but comprehensive — target 800-1000 words.",
        "Use tables, bullet points, and headers for readability.",
    ],
    markdown=True,
)

# ── Level 2: Manager Teams ─────────────────────────────────

data_gathering_team = Team(
    name="Data Gathering Manager",
    mode="parallel",  # Workers run in parallel within this sub-team
    model=model,
    members=[web_researcher, statistics_collector],
    instructions=[
        "Coordinate web research and statistics collection simultaneously.",
        "Merge results into a unified data package for the analysis team.",
    ],
    markdown=True,
)

analysis_team = Team(
    name="Analysis Manager",
    mode="parallel",  # Trend + competitive analysis run in parallel
    model=model,
    members=[trend_analyst, competitive_analyst],
    instructions=[
        "Perform trend and competitive analysis simultaneously.",
        "Synthesize both analyses into unified insights.",
    ],
    markdown=True,
)

synthesis_team = Team(
    name="Synthesis Manager",
    mode="coordinate",
    model=model,
    members=[report_writer],
    instructions=[
        "Take all gathered data and analyses and produce the final executive report.",
    ],
    markdown=True,
)

# ── Level 1: Root Agent ────────────────────────────────────

research_director = Team(
    name="Research Director (Root Agent)",
    mode="coordinate",  # Root coordinator uses AI model to decompose and delegate
    model=model,
    members=[data_gathering_team, analysis_team, synthesis_team],
    instructions=[
        "You are the root orchestrator for complex research projects.",
        "Decompose the research question into: data gathering, analysis, and synthesis.",
        "Delegate to the appropriate sub-team for each phase.",
        "Ensure the final report integrates all findings cohesively.",
        "The flow MUST be: Data Gathering → Analysis → Synthesis (sequential phases).",
    ],

    markdown=True,
)

print("📚 Starting hierarchical research pipeline...")
print("This demonstrates 3 levels: Root → Manager → Worker")
print("=" * 65)

research_director.print_response(
    "Conduct a comprehensive research report on the Agentic AI market in 2025: "
    "market size, key players, emerging trends, and opportunities for startups.",
    stream=True,
)

📚 Starting hierarchical research pipeline...
This demonstrates 3 levels: Root → Manager → Worker


Output()

### Key Takeaways

1. **Hierarchical ⊃ Coordinator**: Hierarchical is an extension of the coordinator pattern across multiple levels.
2. **Mixed patterns**: Each level can use different patterns (root=coordinate, managers=parallel, workers=specialized).
3. **Context engineering matters**: Each level needs to pass relevant context to the next level.
4. **Use sparingly**: Only when the problem genuinely requires multi-level decomposition — the cost and complexity is high.

👉 **Next**: `2_swarm_pattern.ipynb` — decentralized, collaborative agent swarms.